In [1]:
# ── CELL 1: Install ───────────────────────────────────────────────────────────
!pip install transformers -q

In [2]:
# ── CELL 2: Imports ───────────────────────────────────────────────────────────
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import re
import random
import pandas as pd
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

random.seed(42)
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")



Device: cuda


In [3]:
# ── CELL 3: Load GPT-2 ────────────────────────────────────────────────────────
# gpt2-medium gives better paraphrase quality than base gpt2
# still only ~345M params — fits comfortably on Colab T4
MODEL_NAME = "gpt2-medium"

tokenizer_gpt = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer_gpt.pad_token = tokenizer_gpt.eos_token   # GPT-2 has no pad token by default

model_gpt = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
model_gpt.eval()
print("GPT-2 medium loaded")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 medium loaded


In [4]:
# ── CELL 4: Paraphrase synthesis function ────────────────────────────────────
def sanitize(text: str) -> str:
    """Strip non-ASCII and collapse whitespace before tokenizing."""
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def synthesize_paraphrase(
    text       : str,
    num_variants: int   = 2,
    max_new_tokens: int = 120,    # tokens to generate beyond the prompt
    temperature : float = 0.85,   # controls diversity — 0.7-0.9 is sweet spot
    top_p       : float = 0.92,   # nucleus sampling — keeps only top 92% probability mass
    top_k       : int   = 50,     # limits vocab at each step to top 50 tokens
) -> list[str]:
    """
    Strategy:
    1. Take first ~40% of the summary as a prompt prefix
    2. Let GPT-2 complete/rephrase the rest
    3. Extract only the generated portion (not the prompt)
    This forces GPT-2 to stay on topic while producing varied completions.
    """
    text   = sanitize(text)
    words  = text.split()

    # Use first 40% of words as the conditioning prefix
    prefix_len = max(5, int(len(words) * 0.4))
    prefix     = " ".join(words[:prefix_len])

    inputs = tokenizer_gpt(
        prefix,
        return_tensors="pt",
        truncation=True,
        max_length=200,           # cap prefix tokens
    ).to(device)

    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model_gpt.generate(
            inputs["input_ids"],
            max_new_tokens=max_new_tokens,
            num_return_sequences=num_variants,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            pad_token_id=tokenizer_gpt.eos_token_id,
            repetition_penalty=1.3,    # penalise repeating phrases
            no_repeat_ngram_size=3,
        )

    results = []
    for output in outputs:
        # Decode only the newly generated tokens — drop the prompt prefix
        generated = tokenizer_gpt.decode(
            output[prompt_len:],
            skip_special_tokens=True
        ).strip()

        # Combine prefix + generated to form a full paraphrase
        full = (prefix + " " + generated).strip()

        # Clean up — truncate at last full stop to avoid incomplete sentences
        last_stop = max(full.rfind("."), full.rfind("!"), full.rfind("?"))
        if last_stop > len(prefix):
            full = full[:last_stop + 1]

        results.append(full)

    return results


In [5]:
# ── CELL 5: Sanity check ──────────────────────────────────────────────────────
sample = (
    "The Union Budget 2023 focuses on infrastructure development, "
    "with significant allocations for rural roads and electrification schemes."
)
print("Original:")
print(f"  {sample}\n")
variants = synthesize_paraphrase(sample, num_variants=2)
for i, v in enumerate(variants, 1):
    print(f"Variant {i}:")
    print(f"  {v}\n")


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Original:
  The Union Budget 2023 focuses on infrastructure development, with significant allocations for rural roads and electrification schemes.

Variant 1:
  The Union Budget 2023 focuses on ensuring the economic recovery is sustained in order to deliver public services, boost investment and create jobs."
 The report calls for improving spending power among households as well.

Variant 2:
  The Union Budget 2023 focuses on tax and social affairs; spending priorities are prioritised over fiscal targets.
, where the government proposes to spend 2 per cent of gross domestic product (GDP) more than it takes in as a share or taxes - typically around £1bn by 2018-19 when Labour plans for an extra 3 per 1/2 trillion pounds ($4tn). The budget also seeks "to develop new approaches" that help reduce inequality so people can benefit from higher incomes .



In [6]:
# ── CELL 6: Load and split FIRST ──────────────────────────────────────────────
df = pd.read_csv("training_file.csv")
df = df[["chunk_text", "summary_text"]].dropna().reset_index(drop=True)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42)

train_df = train_df.reset_index(drop=True)

val_df.to_csv("val.csv",   index=False)
test_df.to_csv("test.csv", index=False)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("val.csv and test.csv locked.")

Train: 127 | Val: 16 | Test: 16
val.csv and test.csv locked.


In [7]:
# ── CELL 7: Augment — only summary_text is paraphrased ───────────────────────
# GPT-2 is a language model, not a seq2seq model — it's best suited for
# rephrasing the summary (shorter, cleaner text) rather than the long chunk.
# chunk_text is kept as-is — same source, new summary wording = valid pair.

NUM_AUGMENTS = 2    # 127 → 381 rows

augmented_rows = []
skipped        = 0

for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Synthesizing"):
    chunk   = str(row["chunk_text"])
    summary = str(row["summary_text"])

    try:
        summary_variants = synthesize_paraphrase(summary, num_variants=NUM_AUGMENTS)

        for s in summary_variants:
            augmented_rows.append({
                "chunk_text"  : chunk,    # original chunk kept unchanged
                "summary_text": s,        # new paraphrased summary
                "is_augmented": True,
            })
    except Exception as e:
        skipped += 1
        print(f"  [Row {idx}] Skipped — {str(e)[:80]}")
        if device == "cuda":
            torch.cuda.empty_cache()

train_df["is_augmented"] = False
aug_df    = pd.DataFrame(augmented_rows)
train_aug = pd.concat([train_df, aug_df], ignore_index=True)
train_aug = train_aug.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nOriginal : {len(train_df)}")
print(f"Augmented: {len(aug_df)}")
print(f"Skipped  : {skipped}")
print(f"Total    : {len(train_aug)}")

train_aug.to_csv("train_augmented.csv", index=False)
print("train_augmented.csv saved")


Synthesizing:   0%|          | 0/127 [00:00<?, ?it/s]


Original : 127
Augmented: 254
Skipped  : 0
Total    : 381
train_augmented.csv saved
